In [2]:
import zipfile
import SimpleITK as sitk
import numpy as np
import tempfile
import os
import matplotlib.pyplot as plt
import pandas as pd
import h5py

In [2]:
candidates = pd.read_csv('data/candidates_v2.csv')
candidates

,seriesuid,coordX,coordY,coordZ,class
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,68.420000,-74.480000,-288.700000,0
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-95.209361,-91.809406,-377.426350,0
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-24.766755,-120.379294,-273.361539,0
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-63.080000,-65.740000,-344.240000,0
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,52.946688,-92.688873,-241.067872,0
...,...,...,...,...,...
754970,1.3.6.1.4.1.14519.5.2.1.6279.6001.997611074084...,-33.400000,-64.200000,-115.560000,0
754971,1.3.6.1.4.1.14519.5.2.1.6279.6001.997611074084...,56.236359,70.352400,-203.446236,0
754972,1.3.6.1.4.1.14519.5.2.1.6279.6001.997611074084...,-97.104221,55.738289,-203.879785,0
754973,1.3.6.1.4.1.14519.5.2.1.6279.6001.997611074084...,-65.470000,59.670000,-136.370000,0


In [3]:
zip_path = 'data/subset0.zip'
zip = zipfile.ZipFile(zip_path, 'r')
files = zip.namelist()


In [4]:
mhd_files = [f for f in files if f.endswith(".mhd")]

for i in mhd_files[:10]:  # Just read the first 5 files for demonstration
    print(i)

subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524522225658609808059.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674661221381920536987.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.124154461048929153767743874565.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.126121460017257137098781143514.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.126264578931778258890371755354.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.128023902651233986592378348912.mhd
subset0/1.3.6.1.4.1.14519.5.2.1.6279.6001.129055977637338639741695800950.mhd


In [5]:
def load_ct_from_zip(zip_file, mhd_path):
    """
    Extracts ONE CT scan (.mhd + .raw) into temp folder,
    reads it, then deletes immediately.
    """

    with tempfile.TemporaryDirectory() as tmpdir:

        # extract .mhd
        zip_file.extract(mhd_path, tmpdir)

        # read header to find raw filename
        with zip_file.open(mhd_path) as f:
            content = f.read().decode()

        raw_file = None
        for line in content.split("\n"):
            if "ElementDataFile" in line:
                raw_file = line.split("=")[1].strip()

        raw_path = os.path.join(os.path.dirname(mhd_path), raw_file)

        zip_file.extract(raw_path, tmpdir)

        full_mhd = os.path.join(tmpdir, mhd_path)

        img = sitk.ReadImage(full_mhd)

        return img
    
def preprocess(img):
    arr = sitk.GetArrayFromImage(img).astype(np.float32)

    # clip HU range (lungs)
    arr = np.clip(arr, -1000, 400)

    # normalize
    arr = (arr + 1000) / 1400

    return arr

In [6]:
def extract_cube(volume, center, size=40):
    half = size // 2
    z, y, x = center
    z1, z2 = z - half, z + half
    y1, y2 = y - half, y + half
    x1, x2 = x - half, x + half

    return volume[z1:z2, y1:y2, x1:x2]

In [ ]:
#Create a new H5 file to store patches in
h5 = h5py.File("luna16_patches1.h5", "w")

X = h5.create_dataset(
    "X",
    shape=(0, 40, 40, 40),
    maxshape=(None, 40, 40, 40),
    dtype=np.float16,
    compression="gzip",
    compression_opts=4,
    chunks=(1, 40, 40, 40)
)

y = h5.create_dataset(
    "y",
    shape=(0,),
    maxshape=(None,),
    dtype=np.int8,
    compression="gzip",
    compression_opts=4,
    chunks=(1024,))

In [ ]:
#Dictionary for speed
cand_by_scan = {
    uid: df
    for uid, df in candidates.groupby("seriesuid")
}

#Loop over each CT
for index, mhd in enumerate(mhd_files):
    #CT UID
    uid = os.path.basename(mhd).replace(".mhd", "")

    if uid not in cand_by_scan:
        continue

    #Get image data
    img = load_ct_from_zip(zip, mhd)

    #Preprocess to HU range and normalize
    volume = preprocess(img)

    #Get origin and spacing
    origin = np.array(img.GetOrigin())
    spacing = np.array(img.GetSpacing())

    #get all candidates from scan dictionary
    scan_candidates = cand_by_scan[uid]

    #Loop over candidates
    for _, row in scan_candidates.iterrows():
        #Only take 20 % of the negative classes
        label = int(row["class"])
        if label == 0 and np.random.rand() > 0.1:
            continue
        else:
            #Create unique patch id
            patch_id = f"{uid}_{row['coordX']}_{row['coordY']}_{row['coordZ']}"

            #Convert world coordinates to voxel coordinates
            world = np.array([row.coordX, row.coordY, row.coordZ])

            #Voxel coordinates = (world - origin) / spacing
            voxel = np.round((world - origin) / spacing).astype(int)

            x, y, z = voxel

            #Extracting cube around coordinates
            cube = extract_cube(volume, (z, y, x), size=40)
            if cube is None:
                continue


            #NEED TO SAVE TO H5 FILE OR RAM HERE